<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 60 Interface

The notebook checks that every button, menu, field and setting of 3D-Baukasten 1.3 is in the page of
version 2.0, at the place where 1.3 has it, and that every icon is an SVG. The sources are the forms
of 1.3 in the repository: the menu bar of `3D_bau.frm`, the tool bar `F3.frm` with its thirteen
buttons and their tool tips, the body bar `F2.frm`, the register form `F4.frm` with six tabs in two
rows, the bar `F5.frm` and the window F6, which the thesis shows. The page opens in headless Chromium
with WebGPU on SwiftShader, in the modern look.

In [ ]:
width = 1440
height = 900
look = 'modern'

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, theme=look)
page.open()
check(page.errors() == [], 'the page starts without an error', page.errors())

## 1. The menu bar of Form1

Version 1.3 has the menus Datei, Bearbeiten, Objekte and Ansicht, in this order, and Datei holds Neu,
Öffnen, Speichern, Speichern unter, Schließen and Beenden with the shortcuts Strg+N, Strg+O and
Strg+S. Version 2.0 keeps them in their order, fills the empty menus and adds Hilfe at the end.

In [ ]:
titles = page.texts('#menubar .menu-title')
check(titles == ['Datei', 'Bearbeiten', 'Objekte', 'Ansicht', 'Hilfe'], 'the four menus of 1.3 in their order, Hilfe behind them', titles)
datei = page.texts('[data-menu="datei"] > .menu-popup > .menu-entry', 'data-entry')
original = [e for e in datei if e in ('neu', 'oeffnen', 'speichern', 'speichern-unter', 'schliessen', 'beenden')]
check(original == ['neu', 'oeffnen', 'speichern', 'speichern-unter', 'schliessen', 'beenden'], 'Datei keeps the six entries of 1.3 in their order', datei)
check(datei[:5] == original[:5] and datei[-1] == 'beenden', 'the new entries stand between Schließen and Beenden, so no entry of 1.3 moves', datei)
shortcuts = dict(zip(datei, page.texts('[data-menu="datei"] > .menu-popup > .menu-entry .menu-shortcut')))
check(page.texts('[data-entry="neu"] .menu-shortcut') == ['Strg+N'] and page.texts('[data-entry="oeffnen"] .menu-shortcut') == ['Strg+O'] and page.texts('[data-entry="speichern"] .menu-shortcut') == ['Strg+S'], 'the shortcuts of 1.3')
objekte = page.texts('[data-menu="objekte"] .menu-entry', 'data-body')
check(objekte == ['Kugel', 'Kegel', 'Zylinder', 'Würfel', 'Quader', 'Pyramide', 'Prisma', 'Disc', 'Ebene', 'CSG-Vereinigung', 'CSG-Differenz', 'CSG-Durchschnitt', 'CSG-Summe', 'CSG-Ende', '#Neu', 'Neu_Ende'], 'Objekte offers what the body bar and CB1 offer, as the thesis says the menus do', objekte)

## 2. The tool bar F3

`F3.frm` has thirteen buttons in one column. The first nine carry the tool tips of 1.3 letter by
letter, the last four are blank in 1.3 and host undo, redo, the look and the help in 2.0.

In [ ]:
tips = page.texts('#tb1 .tb-button', 'data-tip')
check(len(tips) == 13, 'thirteen buttons', tips)
check(tips[:9] == ['Position', 'Körper einfügen', 'Bearbeiten', 'Farbe ändern', 'Raytrace', '3D-Zoom', 'Ausgabe', 'Beenden', 'Dia-Show'], 'the tool tips of 1.3', tips)
svg = page.eval("() => [...document.querySelectorAll('#tb1 .tb-button')].map((b) => b.querySelector(':scope > svg') !== null)")
check(all(svg), 'every tool carries an SVG icon', svg)
boxes = [page.rect(f'#tb1 [data-tool="{n}"]') for n in range(1, 14)]
check(all(abs(a[0] - boxes[0][0]) < 1 for a in boxes) and all(b[1] > a[1] for a, b in zip(boxes, boxes[1:])), 'the tools stand in one column, top to bottom')

## 3. The body bar F2

`F2.frm` has nine buttons whose keys are the tool tips: Kugel, Kegel, Zylinder, Würfel, Quader,
Pyramide, Prisma, Disc and Ebene. Disc and Ebene have no picture in 1.3, 2.0 gives them one.

In [ ]:
bodies = page.texts('#tb2 .tb-button', 'data-tip')
check(bodies == ['Kugel', 'Kegel', 'Zylinder', 'Würfel', 'Quader', 'Pyramide', 'Prisma', 'Disc', 'Ebene'], 'the nine bodies of 1.3 in their order', bodies)
svg = page.eval("() => [...document.querySelectorAll('#tb2 .tb-button')].map((b) => b.querySelector(':scope > svg') !== null)")
check(all(svg), 'every body carries an SVG icon', svg)
images = page.eval("() => document.querySelectorAll('img:not(#f9-picture)').length")
check(images == 0, 'no bitmap of 1.3 is shipped, the icons are SVG', images)

## 4. The register form F4

`F4.frm` holds an SSTab with six tabs in two rows: Körper, Oberfläche and Ausgabe in front, Bilder,
Kamera and Einstellung behind. Each tab keeps its controls.

In [ ]:
check(page.texts('#sstab1 .tab-row.back .tab') == ['Bilder', 'Kamera', 'Einstellung'], 'the back row')
check(page.texts('#sstab1 .tab-row.front .tab') == ['Körper', 'Oberfläche', 'Ausgabe'], 'the front row')
controls = {
    'koerper': ['#frame8', '#cb2', '#frame1', '#cb1', '#l1-1', '#t1-1', '#t1-2', '#t1-3', '#l1-4', '#l1-6', '#t1-4', '#t1-5', '#t1-6', '#l1-7', '#t1-7', '#t1-8', '#t1-9', '#btn-einf', '#btn-change', '#btn-clear'],
    'ausgabe': ['#frame5', '#text1', '#tfname', '#bpov-comp', '#bpov-save'],
    'bilder': ['#file1', '#tdatei', '#bbild-anz', '#bbild-beenden', '#cmdia'],
    'kamera': ['#label1-kamera', '#frame6', '#hs-4', '#hst-4', '#hs-1', '#hst-1', '#hs-2', '#hst-2', '#hs-3', '#hst-3', '#start', '#command2', '#btn1-1', '#btn2-1'],
    'einstellung': ['#tpasswd', '#lpasswd', '#frame10', '#tbau-home', '#tpov-home', '#tbsp-home', '#tbsp-def', '#bscreen', '#bdatei-change', '#bdatei-def'],
}
page.entry('Pyramide')  # a mask with every caption of the rows of position and size
for tab, ids in controls.items():
    page.page.click(f'#sstab1 [data-tab="{tab}"]')
    page.settle(150)
    hidden = [i for i in ids if not page.visible(i)]
    check(hidden == [], f'the tab {tab} shows its controls of 1.3', hidden)
page.page.click('#sstab1 [data-tab="koerper"]')
check(page.texts('#frame8 legend') == ['OBJEKTE:'] and page.texts('#frame1 legend') == ['GEOMETRIEDATEN:'], 'the captions of the frames of 1.3')
check(page.texts('#btn-einf, #btn-change, #btn-clear') == ['Einfügen', 'Ändern', 'Löschen'], 'the three buttons of 1.3')
check(page.texts('#label1-kamera') == ['noch in der Testphase!!'], 'the note of the tab Kamera stays')
check(page.texts('#bpov-comp, #bpov-save') == ['Übersetzen', 'Ausgeben'], 'the buttons of the tab Ausgabe')

## 5. The bar F5 and the window F6

`F5.frm` holds, left to right, the fields x, y and z, the 3D zoom with `<` and `>`, the 2D zoom with
its field and `<` and `>`, and the status. F6 holds the list TEXTUR and the frame FARBE with its
button Farbe.

In [ ]:
order = ['#tx', '#ty', '#tz', '#d3-zoomin', '#d3-zoomout', '#tzoom', '#d2-zoomin', '#d2-zoomout', '#tstatus']
lefts = [page.rect(s)[0] for s in order]
check(lefts == sorted(lefts), 'the controls of F5 stand in the order of 1.3', lefts)
check(page.texts('#d3-zoomin, #d3-zoomout, #d2-zoomin, #d2-zoomout') == ['<', '>', '<', '>'], 'the captions < and > of 1.3')
check(page.texts('#frame11 legend') == ['TEXTUR'] and page.texts('#bcolor') == ['Farbe'], 'F6 holds TEXTUR and Farbe')
options = page.texts('#cb3 option')
check(options[0] == 'Keine' and len(options) == 44 and options[1] == 'Chrome_Metal', 'Keine and the 43 textures of 1.3', options[:3])

## 6. The places of the windows

`Form_Set` puts the tool bar at the top left, the body bar under it, Form1 in the middle, F4 at the
right edge with F6 under it, and F5 under Form1. `ViewControl` puts the Kreuzriss at the top left, the
Aufriss at the top right, the Schrägriss at the bottom left and the Grundriss at the bottom right.

In [ ]:
r = {k: page.rect(k) for k in ['#f3', '#f2', '#form1', '#f4', '#f6', '#f5', '#views']}
check(r['#f3'][0] < r['#form1'][0] and r['#f3'][1] <= r['#f2'][1], 'F3 at the top left, F2 under it', r['#f3'], r['#f2'])
check(abs(r['#f2'][0] - r['#f3'][0]) < 2, 'F2 in the column of F3')
check(r['#f4'][0] > r['#form1'][0] + r['#form1'][2] - 2, 'F4 at the right edge', r['#f4'])
check(r['#f6'][1] >= r['#f4'][1] + r['#f4'][3] - 2 and abs(r['#f6'][0] - r['#f4'][0]) < 2, 'F6 under F4', r['#f6'])
check(r['#f5'][1] >= r['#views'][1] + r['#views'][3] - 2 and r['#f5'][0] >= r['#form1'][0] - 1, 'F5 under the views of Form1', r['#f5'])
v = {n: page.rect(f'.view[data-view="{n}"]') for n in range(4)}
check(v[1][0] < v[0][0] and v[1][1] < v[3][1] and v[3][0] < v[2][0] and v[0][1] < v[2][1], 'Kreuzriss top left, Aufriss top right, Schrägriss bottom left, Grundriss bottom right', v)

## 7. Pictures

The page in the light and the dark theme of the modern look.

In [ ]:
path, stats = page.screenshot('60_modern_light')
check(stats['lit'] > 0, 'the worker draws the views', stats)
page.eval("() => bausteine.act('dark')")
page.settle(600)
path, stats = page.screenshot('60_modern_dark')
check(page.eval("() => document.documentElement.dataset.theme") == 'dark', 'the dark theme')
page.eval("() => bausteine.act('dark')")
check(page.errors() == [], 'no error on the way', page.errors())

In [ ]:
page.close()
site.stop()